# 从一条行情看懂 Axiom Data

先顺着读，不用运行。下面保存了已执行的小样本输出；它们来自固定的历史演示数据，**不代表当前全量数据已经验收**。

Data 做的事：把供应商发来的数据原样收下，按统一规则整理，保存成以后还能查回去的版本，再交给研究与回测。

我们只跟着中芯国际（688981.SH）走一遍，日期是 2025-06-10 到 2025-06-13。

**原始响应 → 整理后的数据 → 固定版本 → 按时间读取 → 用途视图**

这里不训练模型、不下单。所有演示单元格只读既有数据；不会抓取、构建、修复或恢复 full bulk。

## 先准备这个小例子

下一格只是打开已经保存好的教学样本。路径和版本号不用记；它们的作用是确保今天和下次打开的是同一份材料。
默认不会打开全量生产 Snapshot。若样本不在本机，会提示缺少什么，不会自行下载。

In [1]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display

# 从仓库根或 notebooks 目录启动；也支持 workspace 的 Jupyter 根目录。
REPO = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "axiom-data")
             if (p / "src/axiom_data/__init__.py").is_file()), None)
if REPO is None:
    raise RuntimeError("请在 axiom-data 仓库根或其 notebooks 目录打开本 notebook。")
sys.path.insert(0, str(REPO / "src"))
from axiom_data import SnapshotReader, FactView, load_raw_batch
from axiom_data.operations import compare_event_projection

DATA_ROOT = Path(os.environ.get("AXIOM_DEMO_DATA_ROOT",
    "/var/lib/axiom-data/forensic/pr7-dm2-20260909-r1/admission-r5/source"))
SNAPSHOT_ID = "snapshot-fa3d8b80c729edd2e139a49c82b3dea868bb5639998ba7cc84f540c0623fe786"
if not (DATA_ROOT / "snapshots" / SNAPSHOT_ID / "manifest.json").is_file():
    raise FileNotFoundError("缺少固定演示样本；可以先阅读已保存输出。迁移样本后用 AXIOM_DEMO_DATA_ROOT 指定其根目录。")
SYMBOL, START, END = "688981.SH", "2025-06-10", "2025-06-13"
CUTOFF = "2025-06-13T23:59:59+08:00"
reader = SnapshotReader(DATA_ROOT, SNAPSHOT_ID)
print("已打开：固定历史样本 / 中芯国际 / 4 个交易日。")
print("样本可读取 ≠ 全量生产数据通过验收。")

已打开：固定历史样本 / 中芯国际 / 4 个交易日。
样本可读取 ≠ 全量生产数据通过验收。


## 1. 供应商原来给了什么？

先看原始回执。文件连同“问了什么、什么时候拿到、哪个接口返回的”一起保存，叫 **RawBatch（一次原始响应）**。

这里读的是以前保存的响应，不是重新向供应商查询。后面若发现单位换错了，可以回到这张原始回执核对。

In [2]:
raw_example = None
for ref in reader.commits["market_daily"].manifest["ordered_raw_batch_refs"]:
    batch = load_raw_batch(DATA_ROOT, ref["raw_batch_id"])
    if batch.manifest["request"].get("endpoint") != "daily":
        continue
    matches = [r for r in json.loads(batch.payload)
               if r.get("ts_code") == SYMBOL and r.get("trade_date") == "20250610"]
    if matches:
        raw_example = matches[0]
        break
if raw_example is None:
    raise RuntimeError("该固定样本缺少例子的原始 daily 回执。")
display(pd.DataFrame([{k: raw_example[k] for k in
    ("ts_code", "trade_date", "close", "vol", "amount")}]))
print("这个来源的 vol 单位是手，amount 单位是千元。")

,ts_code,trade_date,close,vol,amount
0,688981.SH,20250610,82.32,254439.12,2108011.021


这个来源的 vol 单位是手，amount 单位是千元。


## 2. 整理后，研究程序看到什么？

Data 按来源合同转换日期、名称和单位。成交量统一成股，成交额统一成元；不会看到一个数很大，就猜它是万元。

这批整理好的某一类数据叫 **DomainCommit（一个数据类别的固定版本）**。行情、财务、股东分别有自己的版本。

本例中：254,439.12 手变成 25,443,912 股；2,108,011.021 千元变成 2,108,011,021 元。转换由正式构建器完成，notebook 只读取结果。

In [3]:
prices = reader.market_daily([SYMBOL], START, END)
display(pd.DataFrame(prices)[["session", "symbol", "close", "volume_shares", "amount_cny"]])
assert len(prices) == 4
assert prices[0]["close"] == 82.32
assert prices[0]["volume_shares"] == 25443912
assert prices[0]["amount_cny"] == 2108011021.0

,session,symbol,close,volume_shares,amount_cny
0,2025-06-10,688981.SH,82.32,25443912,2.108011e+09
1,2025-06-11,688981.SH,82.91,15435894,1.280828e+09
2,2025-06-12,688981.SH,82.27,14536968,1.197309e+09
3,2025-06-13,688981.SH,82.79,22621715,1.862941e+09


## 3. 为什么还要一个 Snapshot？

一次研究往往同时需要行情、财务、行业等数据。**Snapshot 就是一张固定的版本清单**：这次用哪个行情版本、哪个财务版本、哪个行业版本。

它不用复制全部数据。供应商以后改了一份财报，我们可以发布新财务版本和新清单；旧研究仍按旧清单读取。

下面只显示这个样本包含的数据类别与行数。行数证明“存了多少”，不证明“每个需要的日期都有”。

In [4]:
inventory = [{"数据类别": name, "保存行数": len(commit.rows),
              "格式版本": commit.ref.contract_version}
             for name, commit in sorted(reader.commits.items())]
display(pd.DataFrame(inventory))

,数据类别,保存行数,格式版本
0,adjustment_factors,414,adjustment_factors.v1
1,benchmark_daily,69,benchmark_daily.v1
2,corporate_actions,3,corporate_actions.v1
3,financial_events,29,financial_events.v2
4,forecast_observations,3,forecast_observations.v1
5,holder_count_events,67,holder_count_events.v1
6,industry_membership,8,industry_membership.v2
7,margin_daily,12,margin_daily.v1
8,market_daily,414,market_daily.v1
9,moneyflow_daily,12,moneyflow_daily.v1


## 4. “财报属于哪一年”和“当时能不能知道”是两回事

设想一份年报在 4 月公布、6 月修订。3 月的研究不能读到它；5 月的研究也不能偷偷读到 6 月的修订。
这是 **PIT（按当时可知的信息读取）** 要处理的问题。

下面对同一个真实历史样本、同一个 2025 年截止时刻，比较两种问题：

- **按供应商日期尽力还原**：供应商说当时已经公布了什么？这种历史证据有局限。
- **按本系统实际观察时间**：本系统在那个时刻已经保存了什么？

这个样本的财报是在 2026 年观察到的，所以第二种问题在 2025 年可能得到空结果。空结果不表示公司当年没有财报。

In [5]:
financial = reader.as_of("financial_events", symbols=[SYMBOL],
    knowledge_cutoff=CUTOFF, pit_policy="best_effort_vendor_v1")
observed = reader.as_of("financial_events", symbols=[SYMBOL],
    knowledge_cutoff=CUTOFF, pit_policy="operational_pit_v1")
display(pd.DataFrame([
    {"我们问的问题": "按供应商日期尽力还原", "可见记录数": len(financial)},
    {"我们问的问题": "系统在当时实际观察过", "可见记录数": len(observed)},
]))
display(pd.DataFrame(financial)[["endpoint", "report_period", "vendor_available_at",
    "first_observed_at", "pit_qualification"]].head(4))
assert financial and not observed

,我们问的问题,可见记录数
0,按供应商日期尽力还原,20
1,系统在当时实际观察过,0


,endpoint,report_period,vendor_available_at,first_observed_at,pit_qualification
0,income,2024-09-30,2024-11-08T23:59:59+08:00,2026-09-07T16:03:11.127397+00:00,best_effort
1,income,2024-06-30,2024-08-30T23:59:59+08:00,2026-09-07T16:03:11.022601+00:00,best_effort
2,cashflow,2025-03-31,2025-05-09T23:59:59+08:00,2026-09-07T16:03:12.264841+00:00,best_effort
3,cashflow,2024-03-31,2024-05-10T23:59:59+08:00,2026-09-07T16:03:11.858562+00:00,best_effort


## 5. 同一份事实，为什么还有不同 View？

**View 是给不同用途准备的读取方式。** 它们都得说清自己用了哪个 Snapshot。

| 用途 | 给用户什么 | 这里的含义 |
|---|---|---|
| 研究事实 | 财报、股东、行业、时间和缺失原因 | FactView；不替你设计预测因子 |
| 市场回放 | 未复权价格、停牌、限价、公司行动 | MarketReplayView；不负责模拟成交 |
| Qlib 后端 | 已固定的字段、日历和数值文件 | QlibView；不是新的数据源 |
| 复权价格 | 明确锚点的价格换算结果 | 稳定派生；不是成交价格 |

下面读取样本中**已经建好的**复权 View。需要的 View 没有建好时，读取应报错，不能在背后自动构建。
本例因子为 1，复权数值与原价格一样；这不说明其他证券也如此。

In [6]:
VIEWS = {
    "adjusted": "adjusted-price-5722f89cb9640a0caaeb1c653d6bda330e8ffecd7e5f0987a895634051c2c47b",
    "financial": "pr6-fact-1abe9f256a3c7f501b4596974f4b763b30a2885ee435de0a887b83f0d8c0ed4e",
    "event": "pr7-fact-bbec4919922368305ee96afa507304b78a85d6bdd0d05f76883f917f357cebb5",
}
facts = FactView(DATA_ROOT, SNAPSHOT_ID, adjusted_price_view_id=VIEWS["adjusted"],
                 financial_fact_view_id=VIEWS["financial"])
adjusted = facts.read("adjusted_price", symbols=[SYMBOL], start_session=START,
    end_session=END, price_basis="anchor_adjusted", pit_policy="research_non_pit")
display(pd.DataFrame(adjusted["rows"])[["session", "symbol", "close", "adjustment_state"]])

,session,symbol,close,adjustment_state
0,2025-06-10,688981.SH,82.32,ok
1,2025-06-11,688981.SH,82.91,ok
2,2025-06-12,688981.SH,82.27,ok
3,2025-06-13,688981.SH,82.79,ok


## 6. 股东、资金流和财务从哪里读？

它们也走同一个 Snapshot 和时间规则。下面看股东人数；每条记录同时保留报告期和来源。
无需自己去多个文件夹拼数据，也无需在 notebook 里重写修订选择规则。

In [7]:
holders = reader.as_of("holder_count_events", symbols=[SYMBOL],
    knowledge_cutoff=CUTOFF, pit_policy="best_effort_vendor_v1")
display(pd.DataFrame([{"证券": r["symbol"], "报告期": r["report_period"],
    "事实值": r["values"], "历史证据资格": r["pit_qualification"]} for r in holders]).head(6))

,证券,报告期,事实值,历史证据资格
0,688981.SH,2024-06-30,{'number': 246030.0},best_effort
1,688981.SH,2024-02-29,{'number': 248000.0},best_effort
2,688981.SH,2025-03-31,{'number': 257979.0},best_effort
3,688981.SH,2024-03-31,{'number': 246045.0},best_effort
4,688981.SH,2025-04-25,{'number': None},best_effort
5,688981.SH,2025-02-28,{'number': 248801.0},best_effort


## 7. “读出来了”之后，还要检查什么？

一个文件能打开，和它适合本次研究，是两回事。

例如导出到 Qlib 后，不能只看曲线差不多：日期和证券键、缺失状态、数值都应对得上。
下面调用已有的只读比对服务，检查这只股票、这四天、这三个字段。

通过只能说明这次小范围比对通过，不代表全市场完整，不代表严格 PIT，也不代表策略赚钱。

In [8]:
comparison = compare_event_projection(DATA_ROOT, SNAPSHOT_ID, VIEWS["event"],
    symbols=[SYMBOL], fields=["holder.number", "holder.top10_ratio", "margin.balance"],
    start_session=START, end_session=END)
display(comparison)

{'status': 'PASS',
 'snapshot_id': 'snapshot-fa3d8b80c729edd2e139a49c82b3dea868bb5639998ba7cc84f540c0623fe786',
 'view_id': 'pr7-fact-bbec4919922368305ee96afa507304b78a85d6bdd0d05f76883f917f357cebb5',
 'rows': 4,
 'fields': ['holder.number', 'holder.top10_ratio', 'margin.balance'],
 'start_session': '2025-06-10',
 'end_session': '2025-06-13'}

## 8. 没有数值时，应该怎样理解？

| 场景 | Data 应告诉你的事 |
|---|---|
| 股票尚未上市 | 这段时间不适用；不能补成 0 元股价 |
| 停牌没有成交价格 | 保留有证据的停牌状态；不能伪造正常成交 |
| 财报尚未披露 | 在这个截止时刻没有可见财报 |
| 理论上应有、却没有记录 | 保留缺口，调查来源，必要时阻断 |
| 两个来源的日期不同 | 按字段的正式 authority 解释；不能随便选一个让程序跑通 |

上一轮全量审计发现：15 个复权计划要求的锚点当天没有因子。公司身份仍有效，并不保证当天存在因子。
这项全量工作仍保持暂停，锚点规则尚待裁决。这里的小样本运行不会修复它，也不会恢复它。

## 9. 真正更新数据时，后台按什么顺序做？

**先列出要取的数据 → 保存响应 → 整理并校验 → 组成待验收 Snapshot → 生成所需 View → 验收后才允许正式使用。**

| 你遇到的事情 | 正式操作 | 结果 |
|---|---|---|
| 第一次建立一个数据版本 | bootstrap | 固定输入生成候选版本 |
| 每天得到新增或修订数据 | daily | 在明确旧版本上构建增量 |
| 发现历史单位或映射有问题 | repair | 用固定 Raw 生成修复后的新版本 |
| 需要某种已定义读取格式 | materialize_views | 显式生成 View，不在查询时偷偷生成 |
| 进程中断 | 同一计划的 resume | 验证已完成结果后继续；不把退出码当验收 |

这几个操作复用正式服务。notebook 是解释与检查窗口，不是另一套生产流水线。
未变化的 Canonical 分区可以复用；**跨新 Snapshot 复用未变化的 View 计算，目前仍有执行模型缺口**。
不能笼统理解成“改一条数据就一定只重算一条”。

## 10. 你平时只需要知道哪里？

| 位置 | 要不要经常看 |
|---|---|
| 本 notebook | 先看它，理解流程与例子 |
| `README.md`、`docs/operations/` | 真正操作前看对应说明 |
| `src/axiom_data/` | 需要解释实现时再读 |
| `tests/` | 检查系统有没有保持承诺 |
| `reports/`、workspace 的 `review/` | 历史验证和运行证据；不用当教程顺序阅读 |
| `/var/lib/axiom-data/` | 本机正式数据及已迁移样本，按明确版本读取 |
| workspace 的 `data/` | 历史材料，不能把它误认为当前正式入口 |

旧 `pr6`、`pr7` 等名字有些属于已经发布的版本与兼容接口。它们还被测试或旧数据引用，不能只因名字不好看就删除。

以后使用数据，先回答四个问题：**我用哪个版本？查哪只股票哪段时间？允许知道到什么时候？这些数据的缺失和证据是否满足用途？**

不需要先掌握所有内部类和哈希。

## 附录：本次演示的准确身份

这是留给复核人员的细节。这里固定的是历史教学样本，不会自动切到“最新”。
保存输出是一次执行记录；重新运行仍会核验实际输入。

In [9]:
display({"样本根目录": str(DATA_ROOT), "Snapshot": SNAPSHOT_ID,
    "View 引用": VIEWS, "证券": SYMBOL, "区间": [START, END], "知识截止": CUTOFF,
    "用途": "只读教学与小范围比对；非全量生产验收"})

{'样本根目录': '/var/lib/axiom-data/forensic/pr7-dm2-20260909-r1/admission-r5/source',
 'Snapshot': 'snapshot-fa3d8b80c729edd2e139a49c82b3dea868bb5639998ba7cc84f540c0623fe786',
 'View 引用': {'adjusted': 'adjusted-price-5722f89cb9640a0caaeb1c653d6bda330e8ffecd7e5f0987a895634051c2c47b',
  'financial': 'pr6-fact-1abe9f256a3c7f501b4596974f4b763b30a2885ee435de0a887b83f0d8c0ed4e',
  'event': 'pr7-fact-bbec4919922368305ee96afa507304b78a85d6bdd0d05f76883f917f357cebb5'},
 '证券': '688981.SH',
 '区间': ['2025-06-10', '2025-06-13'],
 '知识截止': '2025-06-13T23:59:59+08:00',
 '用途': '只读教学与小范围比对；非全量生产验收'}